In [ ]:
import os
import sys
# add path to custom functions
module_path = os.path.abspath(os.path.join('.')) #../..
if module_path not in sys.path:
    sys.path.append(module_path+"/scripts/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
#np.set_printoptions(threshold=np.inf) # disable truncation
import metpy.calc as mp
import pandas as pd 
import csv
from scipy.stats import pearsonr

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns
# settings
%config InlineBackend.figure_format = 'retina'

# top level data directory (override with the WORK_DATA_DIR env var; see config/paths.env.example)
dpath0=os.environ.get('WORK_DATA_DIR', '/glade/work/dervlamk')
# save figs here
opath=os.environ.get('WORK_DATA_DIR', '/glade/work/dervlamk')

In [ ]:
#=== SET FILE PATH INFO

files = {}

for sim in ['pi']:
    files[sim] = {}
    for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS']:
        files[sim][varn] = f'{dpath0}/PI/dh.precIsotopes.atm.iPI.nc'
    for varn in ['PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS']:
        files[sim][varn] = f'{dpath0}/PI/o.precIsotopes.atm.iPI.nc'
    for varn in ['PRECC', 'PRECL', 'TS']:
        files[sim][varn] = f'{dpath0}/PI/atm.2d.vars.PI.climo.nc'

for sim in ['lig']:
    files[sim] = {}
    for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS',
                'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS',
                'PRECC', 'PRECL',
                'TS']:
        files[sim][varn] = f'{dpath0}/PaleoCalAdjust/data/nc_files/adjusted_files/{varn}_Amon_CESM1.2_LIG127k_146031-182500_cal_adj.nc'


In [ ]:
#=== LOAD DATA

dat={}
sims=['pi','lig']
varns = ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS', 
         'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS', 
         'PRECC', 'PRECL',
         'TS']

for sim in ['pi']:
    dat[sim]={}
    for varn in varns:
        dat[sim][varn]=xr.open_dataset(files[sim][varn])[varn]
        dat[sim][varn].attrs['original_time_values'] = dat[sim][varn].time
        # update time axis?
        dat[sim][varn]=dat[sim][varn].rename({'time':'month'})
        dat[sim][varn]=dat[sim][varn].assign_coords(month=[1,2,3,4,5,6,7,8,9,10,11,12])
     
for sim in ['lig']:
    dat[sim]={}
    for varn in varns:
        ds=xr.open_dataset(files[sim][varn])[varn]
        # groupby won't work on the raw paleocal adjusted data because the "April" data is listed as Month 5 Day 01. Doing so results in the month vector being only 11 in length
        # Shift by 2 days just to nudge it into the correct month
        ndays=2
        ds['time'] = ds.time - pd.Timedelta(days=ndays)
        # now calculate climatologies
        dat[sim][varn] = ds.groupby("time.month").mean(dim='time')


In [ ]:
#=== PROCESS DATA

dDp={}
d18Op={}
prec={}
precc={}
precl={}
sims=['pi','lig']

for sim in sims:
    ptiny=1e-18;
    
    ## Precipitation
    precc[sim] = dat[sim]['PRECC']*1000*60*60*24
    precl[sim] = dat[sim]['PRECL']*1000*60*60*24
    # calculate total precip from convective and large-scale prec vars (snow+rain). convert from m/s to mm/day
    prec[sim] = (dat[sim]['PRECC'] + dat[sim]['PRECL'])*1000*60*60*24
    prec[sim].attrs['units'] = 'mm/day'
    prec[sim].attrs['long_name'] = 'total precipitation'
    prec[sim].attrs['source'] = 'PRECC + PRECL'
    # calculate precipitation weights by month
    annual_total_p = prec[sim].sum(dim="month")
    """
    if sim in ['pi']:
        annual_total_p = prec[sim].sum(dim="time")
    if sim in ['lig']:
        annual_total_p = prec[sim].sum(dim="month")
    """
    pWeights = prec[sim]/annual_total_p
    
    ## Hydrogen
    phyd = dat[sim]['PRECRC_H2Or'] + dat[sim]['PRECRL_H2OR'] + dat[sim]['PRECSC_H2Os'] + dat[sim]['PRECSL_H2OS']
    pdeu = dat[sim]['PRECRC_HDOr'] + dat[sim]['PRECRL_HDOR'] + dat[sim]['PRECSC_HDOs'] + dat[sim]['PRECSL_HDOS']
    # replace very small ph values with a tiny value
    phyd = phyd.where(phyd > ptiny, ptiny) 
    # turn into per mil notation
    dd = (pdeu/phyd - 1)*1000 
    # Multiply isotope values by weights
    dDp[sim] = dd*pWeights
    
    ## Oxygen
    p16o = dat[sim]['PRECRC_H216Or'] + dat[sim]['PRECRL_H216OR'] + dat[sim]['PRECSC_H216Os'] + dat[sim]['PRECSL_H216OS']
    p18o = dat[sim]['PRECRC_H218Or'] + dat[sim]['PRECRL_H218OR'] + dat[sim]['PRECSC_H218Os'] + dat[sim]['PRECSL_H218OS']
    # replace very small ph values with a tiny value
    p16o = p16o.where(p16o > ptiny, ptiny)
    # turn into per mil notation
    do = (p18o/p16o - 1)*1000 
    # Multiply isotope values by weights
    d18Op[sim] = do*pWeights

ts={}
for sim in ['pi','lig']:
    ts[sim] = dat[sim]['TS']

In [ ]:
#=== Calculate LIG-PI differences

# set start and end month indices
im=5
em=9

dDdiff = dDp['lig'][im:em,:,:].mean(dim="month") - dDp['pi'][im:em,:,:].mean(dim="month")
pdiff = prec['lig'][im:em,:,:].mean(dim="month") - prec['pi'][im:em,:,:].mean(dim="month")
pcdiff = precc['lig'][im:em,:,:].mean(dim="month") - precc['pi'][im:em,:,:].mean(dim="month")
pldiff = precl['lig'][im:em,:,:].mean(dim="month") - precl['pi'][im:em,:,:].mean(dim="month")
tsdiff = ts['lig'][im:em,:,:].mean(dim="month") - ts['pi'][im:em,:,:].mean(dim="month")

In [ ]:
#=== LOAD OTHER RELEVANT DATA

# IMERG precipitation
# BASELINE: 2001-2018 everywhere in this project. This notebook previously used 2011-2018,
# which disagreed with fig1_swna_modern_climate.ipynb (2001-2018) — two observational
# baselines in one paper. Standardised on 2001-2018 (Aug 2025).
# NOTE: the derived climo below has NOT been rebuilt yet. Un-comment the block and run it
# once on Casper (needs obs.imerg.precip.2001-2018.nc under $WORK_DATA_DIR/obs_data/)
# before this cell will load.
ds = xr.open_dataset('imerg.gn.2001-2018.climo.nc').precipitation
"""
# imerg precipitation
filen=f'{dpath0}/obs_data/obs.imerg.precip.2001-2018.nc'
ds = xr.open_dataset(filen).precipitation.transpose('time','lat','lon').groupby("time.month").mean(dim='time') * 24 # convert from mm/hr to mm/day
ds.attrs['units'] = 'mm/day'
ds.attrs['Units'] = 'mm/day'
# Convert to 0:360
nx   = len(ds.lon)
lons = np.linspace(0,360,nx)
ds['lon'] = lons
ds=ds.roll(lon=3600)
ds.attrs['source_filename'] = 'obs.imerg.precip.2001-2018.nc'
ds.to_netcdf('imerg.gn.2001-2018.climo.nc', mode='w')
"""

# ETOPO05 topography
filen = f'{dpath0}/obs_data/obs.etopo5.zsurf.nc'
etopo_full = xr.open_dataset(f'{filen}').ROSE
etopo = etopo_full.where(etopo_full>=0, np.nan) # remove bathymetry
etopoSWNA = etopo_full.sel(ETOPO05_X=slice(235,275), ETOPO05_Y=slice(10,42))

# load proxy timeslice mean values (updated as of Aug 2025)
# moved from proxy_data/ to data/processed/ (Aug 2025); path is relative to the repo root,
# which is where notebooks must be launched from
proxydD = pd.read_csv('data/processed/timeslice_mean_proxy_dDraw.csv')
proxydD

# FIGS

## Paper Fig

In [ ]:
#=== PATTERN CORRELATION

# set domain bounds
lon_min = 235
lon_max = 275
lat_min = 10
lat_max = 42

## dDp and Precip
# clip domain data
a = dDdiff.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).values.flatten()
b = pdiff.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).values.flatten()
# calculate pattern correlation between change in dD and precip fields
r, p = pearsonr(a,b)
print(f"dDp & prec  | pattern correlation: r = {r:.3f}, p = {p:.3e}")

## dDp and Tsurf
# clip domain data
a = dDdiff.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).values.flatten()
b = tsdiff.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).values.flatten()
# calculate pattern correlation between change in dD and surface temp fields
r, p = pearsonr(a,b)
print(f"dDp & Tsurf | pattern correlation: r = {r:.3f}, p = {p:.3e}")

In [ ]:
# Proxy Data
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
ddiff = [10.15, 0.19] # based on raw C30 values, August 2025
#ddiff = proxydD['lig_dD'].values - proxydD['late_holocene_dD'].values # will use this once I fix scp issues and upload new file

# Model Data
lon = dat['pi']['PRECC'].lon
lat = dat['pi']['PRECC'].lat
# var specs
months=['January-February-March', 'June-July-August-September', 'July-August-September']
if im==0:
    t_months=months[0]
elif im==5:
    t_months=months[1]
elif im==6:
    t_months=months[2]
else:
    t_months='<not_defined>'
# plot specs
lw=1
bbox={'boxstyle':'square','fc':'white','ec':'black','alpha':1,'pad':0.2}
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['Precipitation', '$\delta$D$_{precip}$', 'Surface Temperature'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]
# isotopes cmap
icmap=cmo.balance
ivmin=-3
ivmax=3
ilevels=np.linspace(ivmin, ivmax, 25) 
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)
# ts cmap
tcmap,_,_,_=get_settings(field='temp', diff=True)
tvmin=-6
tvmax=6
tlevels=np.linspace(tvmin, tvmax, 25)
tnorm=mpl.colors.BoundaryNorm(tlevels, tcmap.N)
# precip cmap
pcmap,_,_,_=get_settings(field='precip', diff=True)
pvmin=-3
pvmax=3
plevels=np.linspace(pvmin, pvmax, 25)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)



# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12,4), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,.975,t_months+' LIG$-$PI Differences', **text_kw)

# precip
ax[0].text(-85,42, 'r $= -0.513$', fontsize=10, ha='right', va='bottom')
cf1=ax[0].pcolormesh(lon, lat, pdiff, cmap=pcmap, norm=pnorm, transform=trans)
ax[0].scatter(clons, clats, c='k', s=125, alpha=1, transform=trans, zorder=100)

# dDp
cf2=ax[1].pcolormesh(lon, lat, dDdiff, cmap=icmap, norm=inorm, transform=trans)
ax[1].scatter(x=clons, y=clats, c=ddiff,
              cmap=icmap, vmin=-5, vmax=5, alpha=1, edgecolor='k', s=125, transform=trans, zorder=100)
for i in [0,1]:
    ax[1].text(clons[i]-1.3, clats[i], f'{ddiff[i]}‰', c='k', fontsize=10, weight='bold', ha='right', bbox=bbox, zorder=100)

# ts
ax[2].text(-85,42, 'r $= 0.702$', fontsize=10, ha='right', va='bottom')
cf3=ax[2].pcolormesh(lon, lat, tsdiff, cmap=tcmap, norm=tnorm, transform=trans)
ax[2].scatter(clons, clats, c='k', s=125, alpha=1, transform=trans, zorder=100)

for i in [0,1,2]:
    ax[i].text(map_bnds[0]+0.5, map_bnds[3]+0.25, titles[i], **text_kw1)
    ax[i].contour(etopoSWNA.ETOPO05_X, etopoSWNA.ETOPO05_Y, etopoSWNA, 
           levels=np.linspace(800,4000,5), linewidths=0.25, colors='k', transform=trans)
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ring=LinearRing(list(zip([-113., -104, -104, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i in [1,2]:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([0.045, -0.025, 0.3, 0.05])
cbar1 = fig.colorbar(cf1, ticks=[-3,-2,-1,0,1,2,3], orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label('[mm day$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')
    
cbar_ax2 = fig.add_axes([0.365, -0.025, 0.3, 0.05])
cbar2 = fig.colorbar(cf2, ticks=[-3,-2,-1,0,1,2,3], orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label(u'[‰]', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax3 = fig.add_axes([0.69, -0.025, 0.3, 0.05])
cbar3 = fig.colorbar(cf3, ticks=np.arange(-8,8,2), orientation='horizontal', extend='both', cax=cbar_ax3)
cbar3.set_label(u'[°C]', weight='normal', labelpad=5, rotation=0)
cbar3.ax.tick_params(labelsize=10)
for tick in cbar3.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,-0.3, 'For $\mathit{adjusted}$ (PaleoCalAdjust) iCESM1.2 LIG (127ka) output. '
         'Not masked for statistical significance.\nLight contours are ETOPO05 topography, '
         'intervals of 800 m starting at 800 m.\nr values are pattern correlation between change in field and change in dDp.')
#plt.savefig("cesm1.2_LIG-PI_jas_dDp_precip.pdf")

In [ ]:
# Proxy Data
clons=proxydD['lon'].values
clats=proxydD['lat'].values
ddiff = proxydD['lig_dD'].values - proxydD['late_holocene_dD'].values

# Model Data
lon = dat['pi']['PRECC'].lon
lat = dat['pi']['PRECC'].lat
# var specs
months=['January-February-March', 'June-July-August-September', 'July-August-September']
if im==0:
    t_months=months[0]
elif im==5:
    t_months=months[1]
elif im==6:
    t_months=months[2]
else:
    t_months='<not_defined>'
# plot specs
lw=1
bbox={'boxstyle':'square','fc':'white','ec':'black','alpha':1,'pad':0.2}
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['$\Delta$$\delta$D$_{precip}$', '$\Delta$Precipitation'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]
# isotopes cmap
icmap=cmo.balance
ivmin=-3
ivmax=3
ilevels=np.linspace(ivmin, ivmax, 25) 
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)
# precip cmap
pcmap,_,_,_=get_settings(field='precip', diff=True)
pvmin=-3
pvmax=3
plevels=np.linspace(pvmin, pvmax, 25)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,t_months+' LIG$-$PI Differences: ADJUSTED', **text_kw)

# dDp
cf1=ax[0].pcolormesh(lon, lat, dDdiff, cmap=icmap, norm=inorm, transform=trans)
ax[0].scatter(x=clons, y=clats, c=ddiff,
              cmap=icmap, vmin=-5, vmax=5, alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
for i in [0,1]:
    ax[0].text(clons[i]-1.1, clats[i], f'{ddiff[i]:.1f}‰', fontsize=10, weight='bold', ha='right', bbox=bbox, zorder=100)

# precip
cf2=ax[1].pcolormesh(lon, lat, pdiff, cmap=pcmap, norm=pnorm, transform=trans)
ax[1].scatter(clons, clats, c='k', s=150, alpha=1, transform=trans, zorder=100)


for i in [0,1]:
    ax[i].contour(etopoSWNA.ETOPO05_X, etopoSWNA.ETOPO05_Y, etopoSWNA, 
           levels=np.linspace(1600,5600,13), linewidths=0.25, colors='k', transform=trans)
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -104, -104, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i==1:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([0.05, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, ticks=[-3,-2,-1,0,1,2,3], orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label(u'[‰]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax2 = fig.add_axes([0.535, -0.025, 0.45, 0.05])
cbar2 = fig.colorbar(cf2, ticks=[-3,-2,-1,0,1,2,3], orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('[mm day$^{-1}$]', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,-0.175, r'For $\mathit{adjusted}$ (PaleoCalAdjust) iCESM1.2 LIG (127ka) output. Pattern correlation between differences for displayed domain: r $= -0.513$')
#plt.savefig("cesm1.2_LIG-PI_jas_dDp_precip.pdf")

## Precip

### LIG-PI Convective vs. Large-Scale Precip Differences

In [ ]:
# Model Data
lon = dat['pi']['PRECC'].lon
lat = dat['pi']['PRECC'].lat
# var specs

# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['$\Delta$Precipitation', '$\Delta$Resolved Precip', '$\Delta$Convective Precip'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]
# precip cmap
pcmap,_,_,_=get_settings(field='precip', diff=True)
pvmin=-3
pvmax=3
plevels=np.linspace(pvmin, pvmax, 25)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)


# ------------------- #
#      Make Plot      #
# ------------------- #
months=['January-February-March', 'June-July-August-September']
t_months=months[1]

fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12,4), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,t_months+' LIG$-$PI Differences', **text_kw)

# Total
cf1=ax[0].pcolormesh(lon, lat, pdiff, cmap=pcmap, norm=pnorm, transform=trans)
ax[1].pcolormesh(lon, lat, plDiff, cmap=pcmap, norm=pnorm, transform=trans)
ax[2].pcolormesh(lon, lat, pcDiff, cmap=pcmap, norm=pnorm, transform=trans)

for i in [0,1,2]:
    ax[i].contour(etopoSWNA.ETOPO05_X, etopoSWNA.ETOPO05_Y, etopoSWNA, 
           levels=np.linspace(1000,5000,5), linewidths=0.25, colors='k', transform=trans)
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -104, -104, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i in [1,2]:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([1, 0.1, 0.015, 0.8])
cbar1 = fig.colorbar(cf1, ticks=[-3,-2,-1,0,1,2,3], orientation='vertical', extend='both', cax=cbar_ax1)
cbar1.set_label(u'[mm day$^{-1}$]', weight='normal', labelpad=15, rotation=270)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,0, r'For $\mathit{adjusted}$ (PaleoCalAdjust) iCESM1.2 LIG (127ka) output. Not masked for statistical significance. '
        'Thin black contours are ETOPO05 topography from 1 km to 5 km, intervals of 1 km.')
#plt.savefig("cesm1.2_LIG-PI_jas_dDp_precip.pdf")

### Annual Precip Cycle

In [ ]:
lat_min = 18
lat_max = 33
lon_min = 247
lon_max = 256

nam_region = {}

for sim in ['obs']:
    nam_region[sim] = ds.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).mean(dim=['lat','lon'])

for sim in ['pi','lig']:
    nam_region[sim] = prec[sim].sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).mean(dim=['lat','lon'])
    
labels=['IMERG','PI','LIG']
tkw = {'axis': 'both', 'direction':'in', 'labelsize': 'x-large'} 
text_kw={'size': 'xx-large', 'weight': 'bold',  'color': 'k', 'ha':'center','va':'bottom'}
patch_kw = {'ec':'beige', 'lw':1, 'ls':'-', 'fc':'beige', 'alpha':0.5} #, 'hatch':'///'}
legend_prop={'size':'x-large', 'weight':'bold'}
legend_kw={'labelcolor':'linecolor', 'ncols':1, 'frameon':False}
line_cols=['k','peru','#9a0200']
labels=['IMERG','PI','LIG']
idx=[0,1,2,3,4,5,6,7,8,9,10,11]

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(9,6), layout='constrained')
ax.text(5.5, 6.15, 'MONTHLY MEAN RAINFALL [18°N$-$33°N, 113°W$-$105°W]', rotation=0, **text_kw)

for i,sim in enumerate(['obs','pi','lig']):
    ax.plot(idx, nam_region[sim], c=line_cols[i], ls='-', lw=2, label=labels[i])
#ax.plot(idx, pobs_clip, c='k', ls='-', lw=2, label='IMERG')
#ax.plot(idx, ppi_clip, c='peru', ls='-', lw=2, label='PI')
#ax.plot(idx, plig_clip, c='#9a0200', ls='-', lw=2, label='LIG')

# plot patch around JJA
pp=plt.Rectangle((5, 0), 4, 7, zorder=1, label='_Hidden', **patch_kw) 
ax.add_patch(pp) 

ax.set(xlim=[0, 11], ylim=[0,6.1])
#ax.set_xlabel('MONTH', weight='bold', size='x-large')
ax.set_xticks([0,1,2,3,4,5,6,7,8,9,10,11])
ax.set_xticklabels(['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC'])
ax.set_yticks([1,2,3,4,5,6])
ax.set_ylabel('[mm/day]', weight='normal', size='x-large')
ax.tick_params(**tkw)
    
ax.legend(loc=2, prop=legend_prop, **legend_kw)

### Monthly Precip Climatology

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
font_kw={'color':'k', 'weight':'bold', 'size':16, 'horizontalalignment':'center'}
lw=1

# colormap specs
cmap2=plt.colormaps['Blues']
vmin2=0
vmax2=12
levels=np.linspace(vmin2, vmax2, 13)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)

# map specs
lat = prec['pi'].lat
lon = prec['pi'].lon
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]

fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(15,15), subplot_kw={'projection': proj}, layout='constrained')
fig.text(.5,1.025,'iCESM1.2 PI Precip Climo', **font_kw)

rows = [0,0,0,1,1,1,2,2,2,3,3,3]
cols = [0,1,2,0,1,2,0,1,2,0,1,2]
mons = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
idx = ['0','1','2','3','4','5','6','7','8','9','10','11']

for i in range(0,12):
    rown = rows[i]
    coln = cols[i]
    ax[rown,coln].text(((map_bnds[0]+map_bnds[1])/2), map_bnds[3]+0.5, mons[i]+ '[' +idx[i]+ ']', **font_kw)
    cf=ax[rown,coln].pcolormesh(lon, lat, prec['pi'].isel(month=i),
                                cmap=cmap2, norm=norm2, transform=trans)

for i,ax in enumerate(ax.flat):
    # add IMERG climatology contours
    #cs=ax.contour(ds.lon, ds.lat, ds[i], levels=np.linspace(5,16,10), linewidths=0.8, colors='fuchsia', transform=trans)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES, linewidth=0.5)
    ring=LinearRing(list(zip([-113., -104, -104, -113.], [18,  18,  33,  33])))
    ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)

cbar_ax = fig.add_axes([0.1, -0.05, 0.8, 0.025])
cbar = fig.colorbar(cf, orientation='horizontal', extend='max', cax=cbar_ax)
cbar.set_label('[mm day$^{-1}$]', labelpad=5, size=14, rotation=0)
cbar.ax.tick_params(labelsize=14)

## Isotopes

In [ ]:
# Model Data
lon = dat['pi']['PRECC'].lon
lat = dat['pi']['PRECC'].lat
# var specs
im=5 #start month
em=9 #end month
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['PI', 'LIG'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]
# isotopes cmap
icmap=cm.RdYlBu_r #o.solar
ivmin=-12
ivmax=0
ilevels=np.linspace(ivmin, ivmax, 25)
inorm=mpl.colors.BoundaryNorm(ilevels, icmap.N)


# ------------------- #
#      Make Plot      #
# ------------------- #
months=['January-February-March', 'June-July-August-September']
t_months=months[1]

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11,5), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,t_months+' LIG & PI Climatologies: ADJUSTED', **text_kw)

#dDp
ax[0].pcolormesh(lon, lat, dDp['pi'][im:em,:,:].mean(dim="month"), cmap=icmap, norm=inorm, transform=trans)
cf1=ax[1].pcolormesh(lon, lat, dDp['lig'][im:em,:,:].mean(dim="month"), cmap=icmap, norm=inorm, transform=trans)



for i in [0,1]:
    # add core location scatter points
    ax[i].scatter(x=proxydD['lon'], y=proxydD['lat'], c='k', alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
    #
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    #ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -104, -104, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i==1:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([0.05, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, ticks=np.arange(-100,10,2), orientation='horizontal', extend='min', cax=cbar_ax1)
cbar1.set_label(u'[‰]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')


cbar_ax2 = fig.add_axes([0.535, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, ticks=np.arange(-100,10,2), orientation='horizontal', extend='min', cax=cbar_ax2)
cbar1.set_label(u'[‰]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,-0.175, r'For $\mathit{adjusted}$ (PaleoCalAdjust) iCESM1.2 LIG (127ka) output.')
#plt.savefig("cesm1.2_LIG-PI_jas_dDp_precip.pdf")

## Temperature

In [ ]:
# Model Data
lon = dat['pi']['TS'].lon
lat = dat['pi']['TS'].lat
# var specs
im=5 #start month
em=9 #end month
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':14, 'ha':'center', 'va':'bottom'}
text_kw1={'color':'k', 'weight':'bold', 'size':12, 'ha':'left', 'va':'bottom'}
titles=np.array(['PI', 'LIG', 'LIG$-$PI'])
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-135., -85., 10., 42.]
# isotopes cmap
cmap,_,_,_=get_settings(field='temp', diff=False)
vmin=15
vmax=35
levels=np.linspace(vmin, vmax, 21) 
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# precip cmap
dcmap,_,_,_=get_settings(field='temp', diff=True)
dvmin=-6
dvmax=6
dlevels=np.linspace(dvmin, dvmax, 25)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)


# ------------------- #
#      Make Plot      #
# ------------------- #
months=['January-February-March', 'June-July-August-September']
t_months=months[1]

fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12,4), layout='constrained', subplot_kw={'projection': proj})
fig.text(.5,1,t_months+' SURFACE TEMPERATURE', **text_kw)

# Climo
cf1=ax[0].pcolormesh(lon, lat, ts['pi'][im:em].mean(dim='month')-273.15, cmap=cmap, norm=norm, transform=trans)
ax[1].pcolormesh(lon, lat, ts['lig'][im:em].mean(dim='month')-273.15, cmap=cmap, norm=norm, transform=trans)
tsDiff=ts['lig']-ts['pi']
cf2=ax[2].pcolormesh(lon, lat, tsdiff, cmap=dcmap, norm=dnorm, transform=trans)

for i in [0,1,2]:
    ax[i].contour(etopoSWNA.ETOPO05_X, etopoSWNA.ETOPO05_Y, etopoSWNA, 
           levels=np.linspace(1600,5600,13), linewidths=0.25, colors='k', transform=trans)
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], **text_kw1)
    ring=LinearRing(list(zip([-113., -104, -104, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i in [1,2]:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False

cbar_ax1 = fig.add_axes([0.05, -0.025, 0.45, 0.05])
cbar1 = fig.colorbar(cf1, ticks=np.arange(0,40,5), orientation='horizontal', extend='both', cax=cbar_ax1)
cbar1.set_label(u'[°C]', weight='normal', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

cbar_ax2 = fig.add_axes([0.535, -0.025, 0.45, 0.05])
cbar2 = fig.colorbar(cf2, ticks=np.arange(-8,8,2), orientation='horizontal', extend='both', cax=cbar_ax2)
cbar2.set_label('$\Delta$[°C]', weight='normal', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

fig.text(0,-0.175, r'For $\mathit{adjusted}$ (PaleoCalAdjust) iCESM1.2 LIG (127ka) output.')
#plt.savefig("cesm1.2_LIG-PI_jas_dDp_precip.pdf")